## Problem: Text Editor Undo and Redo

You are building the undo and redo feature of a simple text editor.

You are given two strings:

1. `text`: the text currently typed in the editor. Each character was typed as one separate action.
2. `operations`: a sequence of keystrokes, where each character is one of:
   - `u` for **undo**: removes the last character of the text.
   - `r` for **redo**: brings back the most recently undone character.

Apply the operations from left to right and return the final text.

**Rules**
- If there is nothing to undo (the text is empty), `u` does nothing.
- If there is nothing to redo, `r` does nothing.
- Redo only brings back characters removed by an earlier undo, and it restores them in the correct order.

**Examples**

| text | operations | output |
|---|---|---|
| `"Hello"` | `"uuruu"` | `"He"` |
| `"Hello"` | `"uur"` | `"Hell"` |
| `"Hello"` | `"uuurrrr"` | `"Hello"` |
| `"Hi"` | `"uuuu"` | `""` |
| `"Hi"` | `"rr"` | `"Hi"` |

**Tasks**
1. Write a function `apply_operations(text, operations)` that returns the final text.
2. Test it on the examples above, and add at least two test cases of your own, including edge casses.
---

In [ ]:
def apply_operations(text, operations):
    # Two stacks: one holds the typed characters, the other holds undone characters
    typed = list(text)
    undone = []

    for op in operations:
        if op == "u":
            if typed:                        # nothing to undo if text is empty
                undone.append(typed.pop())   # last typed char moves to redo stack
        elif op == "r":
            if undone:                       # nothing to redo if no undos happened
                typed.append(undone.pop())   # most recently undone char comes back

    return "".join(typed)

In [ ]:
# Test on the given examples
examples = [
    ("Hello", "uuruu", "He"),
    ("Hello", "uur", "Hell"),
    ("Hello", "uuurrrr", "Hello"),
    ("Hi", "uuuu", ""),
    ("Hi", "rr", "Hi"),
]

for text, operations, expected in examples:
    result = apply_operations(text, operations)
    status = "PASS" if result == expected else "FAIL"
    print(f"{status}: apply_operations({text!r}, {operations!r}) = {result!r}, expected {expected!r}")

In [ ]:
# My own test cases, including edge cases
my_tests = [
    ("", "", ""),                # empty text, no operations
    ("", "uurr", ""),            # empty text: undo and redo both do nothing
    ("abc", "", "abc"),          # no operations: text unchanged
    ("abc", "uuu", ""),          # undo everything exactly
    ("abc", "uuurrr", "abc"),    # undo everything, then redo everything
    ("abc", "ururur", "abc"),    # alternating undo/redo returns to the start
    ("abc", "uurur", "ab"),      # redo restores characters in the correct order
    ("abcd", "uuuuuuurr", "ab"), # extra undos on empty text are ignored, redo still works
]

for text, operations, expected in my_tests:
    result = apply_operations(text, operations)
    status = "PASS" if result == expected else "FAIL"
    print(f"{status}: apply_operations({text!r}, {operations!r}) = {result!r}, expected {expected!r}")

### Why two stacks?

- **Undo** always removes the *last* character typed, which is Last-In-First-Out, so the text works as a stack.
- **Redo** always restores the *most recently undone* character, which is also LIFO, so undone characters go on a second stack.
- `u`: pop from `typed` and push onto `undone` (skip if `typed` is empty).
- `r`: pop from `undone` and push back onto `typed` (skip if `undone` is empty).

Trace for `"Hello"`, `"uuruu"`:

| op | typed | undone |
|---|---|---|
| start | `Hello` | `[]` |
| u | `Hell` | `[o]` |
| u | `Hel` | `[o, l]` |
| r | `Hell` | `[o]` |
| u | `Hel` | `[o, l]` |
| u | `He` | `[o, l, l]` |

**Complexity:** each operation is an O(1) push or pop, so the whole thing is O(n + m) time for text length n and m operations.